# WaferSense – Colab runner

Runs training code from the repo on a Colab GPU. All logic lives in `src/`; this notebook only sets up and calls it.

**Once:** upload these 4 files from `data/processed/` to `MyDrive/Machine Learning/wafer_sense/processed/`:
`wafers.parquet`, `split.parquet`, `maps64_labeled.npy`, `maps64_labeled_ids.npy`.

**Runtime:** Runtime → Change runtime type → T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Machine Learning/wafer_sense'  # path has spaces: quote it in shell commands

## Get the latest code

In [ ]:
import os
if os.path.exists('/content/wafer-sense'):
    !git -C /content/wafer-sense pull
else:
    !git clone https://github.com/Iditc/wafer-sense.git /content/wafer-sense
%cd /content/wafer-sense

## Copy data to local disk
Reading from local disk is much faster than reading from Drive during training.

In [ ]:
!mkdir -p data/processed
!cp -n "{DRIVE}"/processed/* data/processed/
!ls -lh data/processed

In [ ]:
import torch
print(torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Train
Set `MODULE` to the script to run and `ARGS` to its options. Use a new `--name` per run so results are not overwritten.

In [ ]:
# Single run (skip with Run all; uncomment to use)
MODULE = 'src.models.train_cnn'
ARGS = '--name cnn_test'  # e.g. '--name cnn_noweights --no-weights'
# !python -m {MODULE} {ARGS}

## Day 3: one-change experiments
Each run changes one setting from the baseline (64, class weights, 2 channels, seed 42).
First build the 32 and 96 map caches (a few minutes, skipped if they exist). Results are copied to Drive after every run, so a disconnect loses at most one run.

In [ ]:
import os
# Build a cache only if a day 3 run that needs it is still missing on Drive
for size, run in ((32, 'cnn_size32'), (96, 'cnn_size96')):
    done = os.path.exists(f'{DRIVE}/results/metrics/{run}.json')
    if not done and not os.path.exists(f'data/processed/maps{size}_labeled.npy'):
        !python -m src.data.cache_maps --size {size}

In [ ]:
RUNS = {
    'cnn_size32':    '--size 32',
    'cnn_size96':    '--size 96',
    'cnn_noweights': '--no-weights',
    'cnn_1channel':  '--channels 1',
    'cnn_seed1':     '--seed 1',
}
for name, args in RUNS.items():
    if os.path.exists(f'{DRIVE}/results/metrics/{name}.json'):
        print(f'skip {name}: already on Drive')  # lets a rerun resume after a disconnect
        continue
    print(f'===== {name} =====')
    !python -m src.models.train_cnn --name {name} {args}
    !mkdir -p "{DRIVE}/results" "{DRIVE}/checkpoints"
    !cp -r results/* "{DRIVE}/results/"
    !cp -r checkpoints/* "{DRIVE}/checkpoints/" 2>/dev/null || true

## Day 4: autoencoder
Trained on O.K. wafers only; code sizes 128 (main), 32 and 512. Finished runs are skipped, so after a disconnect just run all again.

In [ ]:
AE_RUNS = {
    'ae_code128': '--code-dim 128',
    'ae_code32':  '--code-dim 32',
    'ae_code512': '--code-dim 512',
}
for name, args in AE_RUNS.items():
    if os.path.exists(f'{DRIVE}/results/metrics/{name}.json'):
        print(f'skip {name}: already on Drive')
        continue
    print(f'===== {name} =====')
    !python -m src.models.train_ae --name {name} {args}
    !mkdir -p "{DRIVE}/results" "{DRIVE}/checkpoints"
    !cp -r results/* "{DRIVE}/results/"
    !cp -r checkpoints/* "{DRIVE}/checkpoints/" 2>/dev/null || true

## Save results to Drive
Colab disk is wiped when the session ends.

In [ ]:
!mkdir -p "{DRIVE}/results" "{DRIVE}/checkpoints"
!cp -r results/* "{DRIVE}/results/"
!cp -r checkpoints/* "{DRIVE}/checkpoints/" 2>/dev/null || true